# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and supporting libraries are installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access basic metadata (see note on treating dataset.metadata as an object, not as a dict)
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")


## 2. Data Overview
Review available record sets, fields, and their IDs as defined by Croissant's schema.

**Note:** All references use the `@id` fields for full interoperability and reproducibility.

In [ ]:
# List available record sets by @id
from pprint import pprint

record_sets = list(dataset.record_sets)
print("Record Sets (@id):\n-------------------")
for rs in record_sets:
    print(f"- @id: {rs['@id']}  |  name: {rs.get('name', 'N/A')}")

# Show available fields for each record set by @id
def print_fields_for_record_set(record_set_dict):
    if 'field' in record_set_dict:
        fields = record_set_dict['field']
        # handle single field or list
        if isinstance(fields, dict):
            fields = [fields]
        print(f"  Fields for RecordSet {record_set_dict['@id']}:")
        for fld in fields:
            print(f"    - @id: {fld['@id']}, name: {fld.get('name', 'N/A')}, dataType: {fld.get('dataType', 'N/A')}")

for rs in record_sets:
    print_fields_for_record_set(rs)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the previous overview.

In [ ]:
# For demonstration, pick the first record set and extract its data
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df

# Preview the first record set's columns and data
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets with records found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
This can include removing outliers, transforming data distributions, or grouping data by key attributes for further analysis.

In [ ]:
# Example EDA: operate on a numeric field by @id
# First, auto-detect numeric fields in the main record set
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to detect numeric columns (float or integer dtype)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields available: {numeric_fields}")
    if numeric_fields:
        numeric_field_id = numeric_fields[0]

        # Filter data for values > threshold
        threshold = df[numeric_field_id].quantile(0.75) if df[numeric_field_id].nunique() > 10 else df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, col_norm]].head())

        # Attempt to group by first non-numeric field
        group_fields = [col for col in df.columns if col not in numeric_fields]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"\nGrouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
        else:
            print("No categorical fields found for grouping.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No main record set detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the main numeric field
if main_record_set_id and numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_fields:
        plt.figure(figsize=(12, 5))
        # Only plot if group field not too high in cardinality
        top_group_vals = df[group_field_id].value_counts().index[:5]
        sns.boxplot(data=df[df[group_field_id].isin(top_group_vals)], x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id} (Top 5 groups)')
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, examine, and perform basic data exploration on a Croissant-compliant dataset using `mlcroissant`.

- Used the Croissant schema and referenced all dataset elements by their `@id`.
- Enumerated all available record sets and their fields.
- Loaded data into DataFrames for analysis and created visualizations for numeric fields.
- Illustrated a standard data science workflow reproducible for any Croissant-compatible dataset.

For further analysis, use domain-specific knowledge to select fields and build models, or consult the dataset's online FAIR documentation for scientific context.